# EasyOCR speed benchmark

Runs many **random 10-frame sequences**: each trial either keeps all base frames or swaps **one random slot** for the numbered image — mimicking conditional jersey OCR in the pipeline.

Assets: `images/base/` (10, no number) and `images/with_number/` (one or more; a random pick per injected trial). See [README.md](README.md).

In [ ]:
from pathlib import Path

EXPERIMENT_DIR = Path.cwd()
BASE_DIR = EXPERIMENT_DIR / "images" / "base"
NUMBER_DIR = EXPERIMENT_DIR / "images" / "with_number"
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
SEQUENCE_LENGTH = 10

LANGUAGES = ["en"]
USE_GPU = None  # None = auto; False = force CPU
WARMUP = True

# Fraction of trials that inject the numbered frame at a random slot
P_HAS_NUMBER = 0.3
N_TRIALS = 50
RANDOM_SEED = 42

In [ ]:
import random
import re
import time
from statistics import mean, stdev

import easyocr
import pandas as pd
from PIL import Image

DIGIT_RE = re.compile(r"\d")

In [ ]:
def list_images(directory: Path) -> list[Path]:
    paths = [
        p
        for p in directory.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]
    return sorted(paths, key=lambda p: p.name.lower())


base_paths = list_images(BASE_DIR)
number_paths = list_images(NUMBER_DIR)

if len(base_paths) != SEQUENCE_LENGTH:
    raise FileNotFoundError(
        f"Expected {SEQUENCE_LENGTH} images in {BASE_DIR}, found {len(base_paths)}"
    )
if not number_paths:
    raise FileNotFoundError(f"Need at least 1 image in {NUMBER_DIR}")

print(f"Base frames ({len(base_paths)}):")
for i, p in enumerate(base_paths):
    with Image.open(p) as im:
        print(f"  [{i}] {p.name}  {im.size[0]}x{im.size[1]}")
print(f"Number pool ({len(number_paths)}) — one chosen at random per injected trial:")
for p in number_paths:
    with Image.open(p) as im:
        print(f"  {p.name}  {im.size[0]}x{im.size[1]}")

In [ ]:
gpu = USE_GPU
if gpu is None:
    try:
        import torch

        gpu = torch.cuda.is_available()
    except ImportError:
        gpu = False

print(f"Initializing EasyOCR reader (gpu={gpu}, languages={LANGUAGES})...")
t0 = time.perf_counter()
reader = easyocr.Reader(LANGUAGES, gpu=gpu, verbose=False)
init_s = time.perf_counter() - t0
print(f"Reader ready in {init_s:.2f} s")

In [ ]:
def run_ocr(path: Path) -> list[tuple[str, float]]:
    raw = reader.readtext(str(path), detail=1, paragraph=False)
    return [(str(item[1]).strip(), float(item[2])) for item in raw]


def texts_joined(detections: list[tuple[str, float]]) -> str:
    return " | ".join(t for t, _ in detections if t)


def has_digit(detections: list[tuple[str, float]]) -> bool:
    return any(DIGIT_RE.search(t) for t, _ in detections)


def build_sequence(rng) -> tuple[list[Path], bool, int | None, Path | None]:
    """Return (frame paths, injected?, slot index, chosen number image or None)."""
    slots = list(base_paths)
    if rng.random() >= P_HAS_NUMBER:
        return slots, False, None, None
    idx = rng.randrange(SEQUENCE_LENGTH)
    chosen = rng.choice(number_paths)
    slots[idx] = chosen
    return slots, True, idx, chosen

In [ ]:
if WARMUP:
    print(f"Warmup on {base_paths[0].name}...")
    _ = run_ocr(base_paths[0])
    print("Warmup done.")

In [ ]:
rng = random.Random(RANDOM_SEED)
rows: list[dict] = []

for trial in range(N_TRIALS):
    sequence, injected, inject_slot, number_image = build_sequence(rng)
    trial_start = time.perf_counter()
    for slot, path in enumerate(sequence):
        t0 = time.perf_counter()
        detections = run_ocr(path)
        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        is_number_slot = injected and slot == inject_slot
        rows.append(
            {
                "trial": trial,
                "slot": slot,
                "injected": injected,
                "inject_slot": inject_slot,
                "number_image": number_image.name if number_image else None,
                "is_number_slot": is_number_slot,
                "image": path.name,
                "elapsed_ms": round(elapsed_ms, 2),
                "text": texts_joined(detections) or "(none)",
                "has_digit": has_digit(detections),
                "n_boxes": len(detections),
            }
        )
    trial_ms = (time.perf_counter() - trial_start) * 1000.0
    for r in rows:
        if r["trial"] == trial:
            r["trial_total_ms"] = round(trial_ms, 2)

df = pd.DataFrame(rows)
print(f"Ran {N_TRIALS} trials × {SEQUENCE_LENGTH} frames = {len(df)} OCR calls")
df.head(12)

In [ ]:
def trial_detection_summary(trial_df: pd.DataFrame) -> dict:
    detected = trial_df.loc[trial_df["has_digit"], ["slot", "image", "text"]]
    inject_row = trial_df.loc[trial_df["is_number_slot"]]
    meta = trial_df.iloc[0]
    return {
        "trial": int(meta["trial"]),
        "had_number_injected": bool(meta["injected"]),
        "number_image": meta["number_image"],
        "inject_slot": meta["inject_slot"],
        "ocr_read_as_number": detected["image"].tolist(),
        "ocr_text": [f"{r.image}: {r.text}" for r in detected.itertuples()],
        "inject_slot_detected": bool(inject_row["has_digit"].any()) if len(inject_row) else False,
    }


trial_rows = [trial_detection_summary(df[df["trial"] == t]) for t in range(N_TRIALS)]
trial_report = pd.DataFrame(trial_rows)

n_injected = int(trial_report["had_number_injected"].sum())
n_plain = N_TRIALS - n_injected
hits = int(trial_report.loc[trial_report["had_number_injected"], "inject_slot_detected"].sum())
trial_totals = df.groupby("trial")["trial_total_ms"].first()

all_detected = (
    df.loc[df["has_digit"], "image"]
    .value_counts()
    .rename("times_ocr_saw_digits")
    .reset_index()
    .rename(columns={"image": "image"})
)

times = df["elapsed_ms"].tolist()
times_no_number = df[~df["is_number_slot"]]["elapsed_ms"].tolist()
times_number_slot = df[df["is_number_slot"]]["elapsed_ms"].tolist()

print("--- configuration ---")
print(f"P_HAS_NUMBER:    {P_HAS_NUMBER}")
print(f"N_TRIALS:        {N_TRIALS}")
print(f"RANDOM_SEED:     {RANDOM_SEED}")
print(f"Reader init:     {init_s:.2f} s")
print(f"GPU:             {gpu}")
print(f"Warmup:          {WARMUP}")
print()
print("--- trial mix ---")
print(f"Trials WITH number injected: {n_injected} / {N_TRIALS}")
print(f"Trials WITHOUT number:       {n_plain} / {N_TRIALS}")
if n_injected:
    print(f"Inject slot read as number:  {hits} / {n_injected}")
print()
print("--- images OCR read as having digits (any trial) ---")
if len(all_detected):
    print(all_detected.to_string(index=False))
else:
    print("(none)")
print()
print("--- per-trial: injection vs OCR detection ---")
for _, r in trial_report.iterrows():
    if r["had_number_injected"]:
        planned = f"injected {r['number_image']} at slot {int(r['inject_slot'])}"
    else:
        planned = "no number injected"
    if r["ocr_read_as_number"]:
        seen = ", ".join(r["ocr_read_as_number"])
        ocr = f"OCR saw digits in: {seen}"
    else:
        ocr = "OCR saw digits in: (none)"
    hit = ""
    if r["had_number_injected"]:
        hit = " — inject slot detected" if r["inject_slot_detected"] else " — inject slot NOT detected"
    print(f"trial {r['trial']:3d}  {planned}  |  {ocr}{hit}")
print()
print("--- per-frame latency (all OCR calls) ---")
print(f"Mean:            {mean(times):.1f} ms")
if len(times) > 1:
    print(f"Std dev:         {stdev(times):.1f} ms")
print(f"Min / max:       {min(times):.1f} / {max(times):.1f} ms")
print(f"Effective FPS:   {1000.0 / mean(times):.2f}")
if times_no_number:
    print(f"Mean (no-number slots): {mean(times_no_number):.1f} ms  (n={len(times_no_number)})")
if times_number_slot:
    print(f"Mean (number slot):     {mean(times_number_slot):.1f} ms  (n={len(times_number_slot)})")
print()
print("--- per-sequence (10 frames) ---")
print(f"Mean batch time: {mean(trial_totals):.1f} ms  ({mean(trial_totals) / 1000:.2f} s)")
if len(trial_totals) > 1:
    print(f"Std dev:         {stdev(trial_totals):.1f} ms")
print(f"Batch FPS:       {10000.0 / mean(trial_totals):.2f}")

trial_report

In [ ]:
# Tables: trials with / without injected number, and false positives on base frames
injected_trials = trial_report[trial_report["had_number_injected"]]
plain_trials = trial_report[~trial_report["had_number_injected"]]

print("Trials WITH number injected:")
display(
    injected_trials[
        ["trial", "number_image", "inject_slot", "inject_slot_detected", "ocr_read_as_number", "ocr_text"]
    ]
)

print("Trials WITHOUT number injected:")
display(
    plain_trials[["trial", "ocr_read_as_number", "ocr_text"]]
)

# Base frames that OCR still flagged as having digits (false positives)
base_names = {p.name for p in base_paths}
false_positives = df[df["has_digit"] & df["image"].isin(base_names)][["trial", "slot", "image", "text"]]
if len(false_positives):
    print("Base frames OCR read as having digits (possible false positives):")
    display(false_positives.drop_duplicates(subset=["image"]))
else:
    print("No base frames were read as having digits.")